# 05 — Build, Test & Deploy the Agent

Ties everything together. The agent code lives in **`agent.py`** (loaded via
MLflow *models-from-code*). Here we:

1. Test the agent locally in this notebook.
2. Log it to MLflow with all Databricks **resources** declared (so Model Serving
   can auto-provision credentials).
3. Register it to **Unity Catalog** and deploy it to **Model Serving** with the
   Agent Framework — which also gives you the AI Playground and a Review App.

In [0]:
%pip install -U -r requirements.txt
dbutils.library.restartPython()

In [0]:
%pip install --upgrade --force-reinstall langgraph langchain
dbutils.library.restartPython()

In [0]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"

# LLM — must match the flags at the top of agent.py.
# Default: OpenAI gpt-5.4 via an AI Gateway external-model endpoint (see openai.md).
LLM_PROVIDER = "databricks"     # "databricks" (endpoint) | "openai" (direct)
LLM_ENDPOINT = "openai-chat"    # the external-model endpoint created in openai.md
OPENAI_MODEL = "gpt-5.4"        # used only when LLM_PROVIDER == "openai"

EMBEDDING_ENDPOINT = "databricks-gte-large-en"
VS_INDEX = f"{CATALOG}.{SCHEMA}.research_docs_index"
UC_FUNCTIONS = [
    f"{CATALOG}.{SCHEMA}.get_top_holdings",
    f"{CATALOG}.{SCHEMA}.get_portfolio_positions",
    f"{CATALOG}.{SCHEMA}.get_ticker_exposure",
]
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.financial_intelligence_agent"

# How the structured tools execute the UC functions:
#   "sql_warehouse" (default) -> Statement Execution API (no databricks-connect)
#   "uc_toolkit"              -> UCFunctionToolkit (needs databricks-connect on serverless)
STRUCTURED_TOOLS = "sql_warehouse"
SQL_WAREHOUSE_ID = ""  # leave empty to auto-discover the workspace's SQL warehouse

USE_MCP_SERVICE = False
MCP_SERVICE_NAME = f"{CATALOG}.{SCHEMA}.bigdata_mcp"
USE_GENIE = False
GENIE_SPACE_ID = ""

## 1. Test the agent locally

For the **direct** MCP path, make the API key available to `agent.py` in this
session. (On the deployed endpoint we inject it as a secret-backed env var in step 3.)

In [0]:
import os
os.environ["BIGDATA_API_KEY"] = dbutils.secrets.get(scope="bigdata", key="api_key")

In [0]:
from agent import AGENT
from IPython.display import display, Markdown
response = AGENT.predict(
    {"messages": [{"role": "user",
                   "content": "What are our top 3 holdings by market value?"}]}
)

display(Markdown(response.messages[-1].content))

Top 3 holdings by market value from internal portfolio data:

1. **NVDA — NVIDIA Corporation:** **$17.51M**
2. **MSFT — Microsoft Corporation:** **$9.56M**
3. **AAPL — Apple Inc.:** **$7.41M**

For context, unrealized P&L on those positions is:

- **NVDA:** $7.95M
- **MSFT:** $2.63M
- **AAPL:** $1.40M

These figures are from **internal portfolio holdings data**.

Trace(trace_id=tr-e20b74f035aa48787967b3f08c5c3b89)

### The money shot — one question, all three data sources

In [0]:
# Reload agent module to pick up any code changes
#import sys
#if 'agent' in sys.modules:
#    del sys.modules['agent']
#from agent import AGENT

response = AGENT.predict({"messages": [{"role": "user", "content": (
    "What is our total NVDA exposure across all portfolios? "
    "Summarize our internal investment thesis on NVIDIA, "
    "then compare it with the latest NVIDIA news from Bigdata.com."
)}]})


In [0]:

display(Markdown(response.messages[-1].content))

Here’s a combined view using **internal portfolio data**, **internal research**, and **external Bigdata.com intelligence**.

## 1) Total NVDA exposure across all portfolios
**Internal portfolio data**

| Portfolio | Shares | Market Value | Unrealized P&L |
|---|---:|---:|---:|
| PF002 | 12,000 | $10.506M | $5.106M |
| PF003 | 8,000 | $7.004M | $2.844M |

**Total NVDA exposure across all portfolios**
- **Shares:** **20,000**
- **Market value:** **$17.510M**
- **Unrealized P&L:** **$7.950M**

## 2) Our internal investment thesis on NVIDIA
**Internal research summary**

Our internal view is clearly constructive and currently aligned with an **overweight / strong buy** stance.

### Core bullish points
- **Data center demand remains the main driver**: internal thesis cites **$18.4B data center revenue (+409% YoY)**, driven by strong H100/H200 demand.
- **Product roadmap remains a key support**: confidence is anchored in the **Blackwell / B100-B200 cycle**, with expectation of a major performance step-up.
- **Software moat is central**: the thesis highlights CUDA’s **4M+ developer ecosystem**, reinforcing switching costs and ecosystem lock-in.
- **Inference is the next leg**: internal research frames enterprise AI inference as a large incremental opportunity, with a **$150B TAM by 2027**.
- **Portfolio strategy also reinforces conviction**: the Q1 2025 strategy memo explicitly recommended **increasing NVDA by +3% weight** because **AI training demand exceeds supply**.

### Main internal risks
- **China export restrictions**: internal risk work flags **20–25% of revenue potentially at risk** from export controls.
- **Competition from AMD**: recognized as the main competitive threat, though internal work still sees AMD as lagging NVIDIA in software/ecosystem.
- **Supply constraints**: strong demand is positive, but fulfillment risk remains.
- **Valuation / AI-bubble risk**: broader tech risk memo warns that AI infrastructure spending may be front-loaded ahead of proven enterprise ROI.

### Bottom line on internal thesis
Internally, NVIDIA is viewed as:
1. **The leading AI infrastructure platform**
2. Protected by a **durable software + ecosystem moat**
3. Supported by a **strong near-term product cycle**
4. But exposed to **valuation sensitivity, geopolitics, and emerging competition**

## 3) Comparison with the latest NVIDIA news from Bigdata.com
**External data from Bigdata.com**  
The latest news flow broadly **supports** our internal thesis, but also sharpens the **competition** and **execution** debate.

### Where the news confirms our thesis
**A. Demand for NVIDIA AI infrastructure remains strong**
- Bristol Myers is expanding compute infrastructure around NVIDIA DGX SuperPOD / Vera Rubin systems, reinforcing the idea that enterprise and industry customers are still scaling AI spend. [The Fly, “Bristol Myers to expand compute infrastructure to deploy Nvidia DGX SuperPOD,” Jul 20, 2026](https://app.bigdata.com/documents/3097E48603D94101F7C6076227B38F29?cnum=1)
- MT Newswires similarly reports Bristol Myers is expanding its NVIDIA partnership to build an AI factory for R&D, another sign of adoption beyond hyperscalers. [MT Newswires, “Bristol-Myers Squibb Expands Partnership With NVIDIA to Build AI 'Factory',” Jul 20, 2026](https://app.bigdata.com/documents/1F995ABD21D557DC0873F6F9470C181D?cnum=1)
- QumulusAI’s purchase of **1,632 NVIDIA Blackwell B300 GPUs** also supports the idea that demand continues to outpace supply. [The Fly, “QumulusAI purchases 1,632 NVIDIA Blackwell B300 GPUs,” Jul 20, 2026](https://app.bigdata.com/documents/FB41650BF809111909B7F78C0F860D8D?cnum=1)

**Read-through:** this is consistent with our internal view that AI training/infrastructure demand is still robust and that the next-generation roadmap matters.

**B. The product/ecosystem story remains active**
- NVIDIA expanded its **Agent Toolkit** with Omniverse libraries to enable physical AI capabilities, which reinforces the internal thesis that NVIDIA is more than a chip vendor and is extending its software/platform moat. [MT Newswires, “Nvidia Expands Agent Toolkit to Help AI Agents Build Physical Capabilities for Applications,” Jul 20, 2026](https://app.bigdata.com/documents/F82F310F40467EA8279896046B1A6801?cnum=1)

**Read-through:** this supports our internal CUDA/platform-moat argument.

### Where the news challenges or complicates our thesis
**A. AMD competition is becoming more credible**
- Benzinga reports AMD unveiled **Helios**, a rack-scale AI system positioned against NVIDIA Grace Blackwell / Vera Rubin, with customers including Microsoft, Meta, OpenAI, and Oracle. The same article notes Futurum estimates NVIDIA still controls **>95%** of the data center GPU market, but AMD could gain share over time. [Benzinga, “What's Going on With NVIDIA Stock Monday?”, Jul 20, 2026](https://app.bigdata.com/documents/AAA934ADA72D6B3020EEBCB7AB1FD309?cnum=1)
- Another Benzinga piece says Microsoft will deploy AMD’s Helios on Azure, underscoring that hyperscalers are actively cultivating alternatives. [Benzinga, “AMD Rises After Microsoft's Helios Win: Is Nvidia Finally Facing Real Competition?”, Jul 20, 2026](https://app.bigdata.com/documents/4EE85DC15E66C61107B1E114FB12B3B4?cnum=1)

**Read-through:** this aligns with our internal risk framework. Our thesis already flags AMD as the main competitive threat; the latest news suggests that threat is moving from theoretical to more tangible.

**B. Partnerships remain strong, but concentration questions remain**
- Bechtle’s AI factory partnership with NVIDIA and Dell points to broader ecosystem expansion in Europe. [Alliance News, “Bechtle teams with Nvidia and Dell on Nuremberg 'AI factory',” Jul 20, 2026](https://app.bigdata.com/documents/A1D8425C8B315F23FB4801EB308A9692?cnum=1)

**Read-through:** positive for commercial momentum, though it does not directly reduce the valuation or geopolitical risks highlighted internally.

## 4) Synthesis: internal thesis vs latest external news
### Overall comparison
The latest **Bigdata.com** news flow is **mostly supportive** of our internal thesis.

**What is confirmed externally**
- Demand for NVIDIA AI systems remains strong
- Blackwell/Vera Rubin class infrastructure is being adopted
- NVIDIA’s moat is still reinforced by software/platform expansion
- The company remains the dominant incumbent in AI infrastructure

**What is getting more important externally**
- AMD is becoming a more visible competitive challenger, especially via large cloud customer wins
- The debate is shifting from “whether competition appears” to “how fast share erosion could happen”

### Investment implication
If we compare internal conviction with current news flow:
- **Bull case remains intact**
- **Near-term demand indicators remain healthy**
- **Competitive risk is rising but not yet thesis-breaking**

So the external tape from **Bigdata.com** currently suggests:
> **Our internal thesis is still directionally right, but the market should watch AMD competition more closely than before.**

## Sources
1. Bigdata.com / Benzinga, “What's Going on With NVIDIA Stock Monday?”, Jul 20, 2026, https://app.bigdata.com/documents/AAA934ADA72D6B3020EEBCB7AB1FD309?cnum=1  
2. Bigdata.com / The Fly, “Bristol Myers to expand compute infrastructure to deploy Nvidia DGX SuperPOD,” Jul 20, 2026, https://app.bigdata.com/documents/3097E48603D94101F7C6076227B38F29?cnum=1  
3. Bigdata.com / MT Newswires, “Bristol-Myers Squibb Expands Partnership With NVIDIA to Build AI 'Factory',” Jul 20, 2026, https://app.bigdata.com/documents/1F995ABD21D557DC0873F6F9470C181D?cnum=1  
4. Bigdata.com / Benzinga, “AMD Rises After Microsoft's Helios Win: Is Nvidia Finally Facing Real Competition?”, Jul 20, 2026, https://app.bigdata.com/documents/4EE85DC15E66C61107B1E114FB12B3B4?cnum=1  
5. Bigdata.com / Alliance News, “Bechtle teams with Nvidia and Dell on Nuremberg 'AI factory',” Jul 20, 2026, https://app.bigdata.com/documents/A1D8425C8B315F23FB4801EB308A9692?cnum=1  
6. Bigdata.com / MT Newswires, “Nvidia Expands Agent Toolkit to Help AI Agents Build Physical Capabilities for Applications,” Jul 20, 2026, https://app.bigdata.com/documents/F82F310F40467EA8279896046B1A6801?cnum=1  
7. Bigdata.com / The Fly, “QumulusAI purchases 1,632 NVIDIA Blackwell B300 GPUs,” Jul 20, 2026, https://app.bigdata.com/documents/FB41650BF809111909B7F78C0F860D8D?cnum=1

If you want, I can next turn this into a **PM-ready one-pager** with **positioning, thesis, catalysts, and risks**.

## 2. Log the agent to MLflow

Declaring **resources** at log time lets Model Serving mint short-lived
credentials for each dependency (the LLM, the vector index, the UC functions, and
— on the governed path — the Genie space and MCP Service).

In [0]:
import mlflow
from importlib.metadata import version
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
    DatabricksServingEndpoint(endpoint_name=EMBEDDING_ENDPOINT),
    DatabricksVectorSearchIndex(index_name=VS_INDEX),
    *[DatabricksFunction(function_name=fn) for fn in UC_FUNCTIONS],
]

# The sql_warehouse strategy executes the UC functions via the Statement Execution API,
# so the served model needs a SQL warehouse resource. Auto-discover one if not set.
if STRUCTURED_TOOLS == "sql_warehouse":
    from mlflow.models.resources import DatabricksSQLWarehouse
    from databricks.sdk import WorkspaceClient

    wh_id = SQL_WAREHOUSE_ID or next(iter(WorkspaceClient().warehouses.list())).id
    print(f"Using SQL warehouse: {wh_id}")
    resources.append(DatabricksSQLWarehouse(warehouse_id=wh_id))

# Governed path: add the MCP Service's resources (derived automatically).
if USE_MCP_SERVICE:
    from databricks.sdk import WorkspaceClient
    from databricks_mcp import DatabricksMCPClient

    ws = WorkspaceClient()
    service_url = f"{ws.config.host}/ai-gateway/mcp-services/{MCP_SERVICE_NAME}"
    resources += DatabricksMCPClient(
        server_url=service_url, workspace_client=ws
    ).get_databricks_resources()

# Optional Genie space resource
if USE_GENIE and GENIE_SPACE_ID:
    from mlflow.models.resources import DatabricksGenieSpace

    resources.append(DatabricksGenieSpace(genie_space_id=GENIE_SPACE_ID))

# Pin the agent's dependencies for the served model. Beyond databricks-langchain you MUST
# include `langchain`, `langchain-core`, `langgraph` AND `langgraph-prebuilt` — MLflow's
# LangGraph ChatAgent helpers import langgraph.prebuilt.ToolNode (its own package on
# LangGraph 0.3.x); missing it surfaces as the generic
# "Please install langchain>=0.2.17 and langgraph>=0.2.0" error.
with mlflow.start_run():
    logged = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            f"databricks-langchain=={version('databricks-langchain')}",
            f"langchain=={version('langchain')}",
            f"langchain-core=={version('langchain-core')}",
            f"langgraph=={version('langgraph')}",
            f"langgraph-prebuilt=={version('langgraph-prebuilt')}",
            f"langchain-mcp-adapters=={version('langchain-mcp-adapters')}",
            f"databricks-mcp=={version('databricks-mcp')}",
            f"databricks-sdk=={version('databricks-sdk')}",
            f"nest-asyncio=={version('nest-asyncio')}",
            f"mlflow=={version('mlflow')}",
        ],
    )

print(logged.model_uri)

## 3. Register to Unity Catalog & deploy to Model Serving

In [0]:
mlflow.set_registry_uri("databricks-uc")
registered = mlflow.register_model(model_uri=logged.model_uri, name=UC_MODEL_NAME)
print(f"Registered {UC_MODEL_NAME} v{registered.version}")

In [0]:
from databricks import agents

# On the DIRECT MCP path, pass the API key to the endpoint as a secret-backed env var
# so agent.py can read os.environ["BIGDATA_API_KEY"] at serving time.
deploy_kwargs = {}
if not USE_MCP_SERVICE:
    deploy_kwargs["environment_vars"] = {
        "BIGDATA_API_KEY": "{{secrets/bigdata/api_key}}"
    }

agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=registered.version,
    scale_to_zero=True,
    tags={"demo": "bigdata-mcp"},
    **deploy_kwargs,
)

## 4. Chat with it

Deployment takes a few minutes. When it's ready:

- Open **Serving** → your endpoint → **Use → AI Playground** to chat, or share the
  **Review App** link with business users for feedback.
- Query it programmatically from anywhere:

```python
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")
client.predict(
    endpoint="agents_bigdata_demo-financial_intelligence-financial_intelligence_agent",
    inputs={"messages": [{"role": "user",
             "content": "Show our AI & Semiconductor portfolio, then get NVIDIA's "
                        "Bigdata.com tearsheet and the latest news."}]},
)
```

### Demo questions to try in the Playground

**Internal structured**
- What are the top 5 holdings by market value across all portfolios?
- What is our total NVDA exposure across all portfolios?
- Show all positions in portfolio PF002.

**Internal unstructured**
- What is our internal investment thesis on NVIDIA?
- What are the key risks in our technology sector assessment?
- What allocation changes does the Q1 2025 strategy memo recommend?

**External (Bigdata.com) — including the newer tools**
- What are analysts saying about Apple's latest earnings? *(bigdata_search)*
- Give me a financial tearsheet for Microsoft. *(find_securities → bigdata_company_tearsheet)*
- What is the current media sentiment on NVIDIA? *(bigdata_sentiment_tearsheet)*
- Which of these companies report earnings in the next two weeks? *(bigdata_events_calendar)*
- Show the holdings and allocations of a major semiconductor ETF. *(bigdata_etf_tearsheet)*
- Give me a US macro and cross-asset market snapshot. *(country/market tearsheet)*

**Cross-source (the reason this demo exists)**
- What is our largest NVDA holding? Compare our internal thesis with the latest
  NVIDIA news from Bigdata.com.
- For our top 5 holdings, pull Bigdata.com media sentiment for each and flag any where
  sentiment is turning negative versus our internal thesis.
- Which of our holdings report earnings in the next two weeks (Bigdata.com events
  calendar)? Summarize the setup for the two largest positions.
- Compare our concentrated NVDA exposure with a semiconductor ETF's holdings via the
  Bigdata.com ETF tearsheet — are we more or less concentrated than the index?
- Screen our holdings for credit-factor risk with Bigdata.com and cross-reference the
  flags with our internal risk assessment.
- What does our risk assessment say about China exposure? Find the latest Bigdata.com
  news on China semiconductor export policy.